In [ ]:
## sequesnce to sequence language translation
#https://keras.io/examples/nlp/lstm_seq2seq/

In [2]:
from keras.models import Model
from keras.layers import Input, LSTM, Dense
import numpy as np

In [3]:
batch_size = 64
epochs = 100
latent_dim = 256
num_samples = 10000
data_path = r"D:\Coding\DeepLearning\Datasets\fra.txt"

In [4]:
## vectorize the data
input_texts = []
target_texts = []
input_characters = set()
target_characters = set()
with open(data_path, 'r', encoding='utf-8') as f:
    lines = f.read().split('\n')

for line in lines[: min(num_samples, len(lines) - 1)]:
    # print(line)
    # print(line.split('\t'), len(line.split('\t')))
    input_text, target_text, _ = line.split('\t')
    target_text = '\t' + target_text + '\n'

    input_texts.append(input_text)
    target_texts.append(target_text)

    for char in input_text:
        if char not in input_characters:
            input_characters.add(char)

    for char in target_text:
        if char not in target_characters:
            target_characters.add(char)


In [5]:

input_characters = sorted(list(input_characters))
target_characters = sorted(list(target_characters))
num_encoder_tokens = len(input_characters)
num_decoder_tokens = len(target_characters)
max_encoder_seq_length = max([len(txt) for txt in input_texts]) 
max_decoder_seq_length = max([len(txt) for txt in target_texts])

In [11]:
print('Number of Samples : ', len(input_texts))
print('Numbber of unique input tokens : ', num_encoder_tokens)
print('Numbber of unique output tokens : ', num_decoder_tokens)
print('Max sequence length for inputs : ', max_encoder_seq_length)
print('Max sequence length for outputs : ', max_decoder_seq_length)
print('Input characters : ', input_characters)
print('Target characters : ', target_characters)

Number of Samples :  10000
Numbber of unique input tokens :  70
Numbber of unique output tokens :  91
Max sequence length for inputs :  14
Max sequence length for outputs :  59
Input characters :  [' ', '!', '"', '$', '%', '&', "'", ',', '-', '.', '0', '1', '2', '3', '5', '7', '8', '9', ':', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'Y', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
Target characters :  ['\t', '\n', ' ', '!', '%', '&', "'", ',', '-', '.', '0', '1', '2', '3', '5', '8', '9', ':', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'Y', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\xa0', '«', '»', 'À', 'Ç', 'É', 'Ê', 'à', 'â', 'ç', 'è', 'é', 'ê', 'î', 'ï', 'ô', 

In [12]:
input_token_index = dict([(char, i) for i, char in enumerate(input_characters)])
target_token_index = dict([(char, i) for i, char in enumerate(target_characters)])

In [ ]:
# input_token_index


In [15]:
encoder_input_data = np.zeros((len(input_texts), max_encoder_seq_length, num_encoder_tokens), dtype='float32')
decoder_input_data = np.zeros((len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype='float32')
decoder_target_data = np.zeros((len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype='float32')

In [16]:
for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
    for t, char in enumerate(input_text):
        encoder_input_data[i, t, input_token_index[char]] = 1.
    encoder_input_data[i, t + 1:, input_token_index[' ']] = 1.
    for t, char in enumerate(target_text):
        # decoder target data is ahead of decoder input data by one timestep
        decoder_input_data[i, t, target_token_index[char]] = 1.
        if t > 0:
            # decoder target data will be ahead by one timestep
            # and will not include the start character.
            decoder_target_data[i, t - 1, target_token_index[char]] = 1.
    decoder_input_data[i, t + 1:, target_token_index[' ']] = 1.
    decoder_target_data[i, t:, target_token_index[' ']] = 1.

In [17]:
encoder_input_data.shape, decoder_input_data.shape, decoder_target_data.shape

((10000, 14, 70), (10000, 59, 91), (10000, 59, 91))

In [18]:
## define an input sequence and process it.
encoder_inputs = Input(shape=(None, num_encoder_tokens))
encoder = LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder(encoder_inputs)
# we discard `encoder_outputs` and only keep the states.
encoder_states = [state_h, state_c]

In [19]:
# setup the decoder, using `encoder_states` as initial state.
decoder_inputs = Input(shape=(None, num_decoder_tokens))
# we set up our decoder to return full output sequences,
# and to return internal states as well. We don't use the
# return states in the training model, but we will use them in inference.
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

In [20]:
# define the model that will turn
# `encoder_input_data` & `decoder_input_data` into `decoder_target_data`
model= Model([encoder_inputs, decoder_inputs], decoder_outputs)

# run training
model.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])

model.fit([encoder_input_data, decoder_input_data], decoder_target_data,
          batch_size=batch_size,
          epochs=epochs,
          validation_split=0.2)


Epoch 1/100


125/125 [==============================] - 45s 298ms/step - loss: 1.2022 - accuracy: 0.7334 - val_loss: 1.0608 - val_accuracy: 0.7181
Epoch 2/100
125/125 [==============================] - 39s 314ms/step - loss: 0.9313 - accuracy: 0.7497 - val_loss: 0.9689 - val_accuracy: 0.7255
Epoch 3/100
125/125 [==============================] - 38s 302ms/step - loss: 0.8480 - accuracy: 0.7665 - val_loss: 0.8734 - val_accuracy: 0.7493
Epoch 4/100
125/125 [==============================] - 37s 298ms/step - loss: 0.7581 - accuracy: 0.7894 - val_loss: 0.7845 - val_accuracy: 0.7790
Epoch 5/100
125/125 [==============================] - 33s 263ms/step - loss: 0.6802 - accuracy: 0.8044 - val_loss: 0.7187 - val_accuracy: 0.7924
Epoch 6/100
125/125 [==============================] - 38s 301ms/step - loss: 0.6335 - accuracy: 0.8156 - val_loss: 0.6861 - val_accuracy: 0.7992
Epoch 7/100
125/125 [==============================] - 35s 277ms/step - loss: 0.6044 - accuracy: 0.8230 - val_loss: 0.659

In [22]:
# inference mode(Sampling)
# Here's the dril:

# 1) encode input and retrieve initial decoder state
# 2) run one step of decoder with this initial state
# and a "start of sequence" token as target.
# Output will be the next target token
# 3) Repeat with the current target token and current states

# Define sampling models
encoder_model = Model(encoder_inputs, encoder_states)

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
decoder_outputs, state_h, state_c = decoder_lstm(decoder_inputs, initial_state=decoder_states_inputs)
decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_outputs)
decoder_model = Model([decoder_inputs] + decoder_states_inputs, [decoder_outputs] + decoder_states)

# Reverse-lookup token index to decode sequences back to
# something readable.   
reverse_input_char_index = dict((i, char) for char, i in input_token_index.items())
reverse_target_char_index = dict((i, char) for char, i in target_token_index.items())

def decode_sequence(input_seq): 
    # Encode the input as state vectors.
    states_value = encoder_model.predict(input_seq)

    # Generate empty target sequence of length 1.
    target_seq = np.zeros((1, 1, num_decoder_tokens))   
    # Populate the first character of target sequence with the start character.
    target_seq[0, 0, target_token_index['\t']] = 1.

    # Sampling loop for a batch of sequences
    # (to simplify, here we assume a batch of size 1).
    stop_condition = False
    decoded_sentence = ''
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)

        # Sample a token
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = reverse_target_char_index[sampled_token_index]
        decoded_sentence += sampled_char

        # Exit condition: either hit max length
        # or find stop character.
        if (sampled_char == '\n' or
           len(decoded_sentence) > max_decoder_seq_length):
            stop_condition = True

        # Update the target sequence (of length 1).
        target_seq = np.zeros((1, 1, num_decoder_tokens))
        target_seq[0, 0, sampled_token_index] = 1.

        # Update states
        states_value = [h, c]
    return decoded_sentence

In [23]:
for seq_index in range(20):
    # Take one sequence (part of the training set)
    # for trying out decoding.
    input_seq = encoder_input_data[seq_index: seq_index + 1]
    decoded_sentence = decode_sequence(input_seq)
    print('-')
    print('Input sentence:', input_texts[seq_index])    
    print('Decoded sentence:', decoded_sentence)

1/1 [==============================] - 0s 44ms/step
-
Input sentence: Go.
Decoded sentence: Achetez-le !

1/1 [==============================] - 0s 41ms/step
-
Input sentence: Go.
Decoded sentence: Achetez-le !

1/1 [==============================] - 0s 46ms/step
-
Input sentence: Go.
Decoded sentence: Achetez-le !

1/1 [==============================] - 0s 35ms/step
-
Input sentence: Go.
Decoded sentence: Achetez-le !

1/1 [==============================] - 0s 36ms/step
-
Input sentence: Hi.
Decoded sentence: Elle guison !

1/1 [==============================] - 0s 42ms/step
-
Input sentence: Hi.
Decoded sentence: Elle guison !

1/1 [==============================] - 0s 50ms/step
-
Input sentence: Run!
Decoded sentence: Fuyez !

1/1 [==============================] - 0s 35ms/step
-
Input sentence: Run!
Decoded sentence: Fuyez !

1/1 [==============================] - 0s 45ms/step
-
Input sentence: Run!
Decoded sentence: Fuyez !

1/1 [==============================] - 0s 40ms/step
-
In